# Multiple Linear Regression Bias Analysis

Uses the prepared analysis_data.csv (one row per person) to test for sentencing disparities between racial groups while controlling for offense history, enhancements, and county.

## Configuration

Sets the two groups to compare and the outcome variable.

In [1]:
exposed = "White"
unexposed = "Black"
outcome_col = "aggregate sentence in months"

## Import Libraries

In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan
import statsmodels.api as sm

## Load Analysis Data

Loads the prepared dataset where each row represents one person with all features pre-calculated.

In [3]:
data = pd.read_csv('analysis_data.csv')

print(f"Total rows: {len(data):,}")
print(f"Total columns: {len(data.columns)}")

Total rows: 95,476
Total columns: 21


## Filter to Comparison Groups

Restricts analysis to Black and White defendants only.

In [4]:
data = data[data['ethnicity'].isin([exposed, unexposed])].copy()

data['ethnicity'] = pd.Categorical(
    data['ethnicity'],
    categories=[exposed, unexposed],
    ordered=False
)

print(f"Sample size: {len(data):,}")
print(data['ethnicity'].value_counts())

Sample size: 45,459
ethnicity
Black    26428
White    19031
Name: count, dtype: int64


## Clean Data

Drops rows missing critical variables needed for regression.

In [5]:
data = data.dropna(subset=[outcome_col, 'ethnicity', 'controlling case sentencing county'])

print(f"Final sample: {len(data):,}")
print(f"\nMissing values per column:")
print(data.isnull().sum()[data.isnull().sum() > 0])

Final sample: 45,459

Missing values per column:
Series([], dtype: int64)


## Build Regression Formula

Dynamically builds the formula including ethnicity, county, all current offense counts, all prior offense counts, and enhancement flags.

In [6]:
prior_cols = [c for c in data.columns if c.startswith('count_prior')]
current_cols = [c for c in data.columns if c.startswith('count_current')]

formula_parts = (
    [f'Q("{outcome_col}")']
    + ['C(ethnicity)']
    + ['C(Q("controlling case sentencing county"))']
    + current_cols
    + prior_cols
)

formula = f"{formula_parts[0]} ~ {' + '.join(formula_parts[1:])}"

print("Formula:")
print(formula)

Formula:
Q("aggregate sentence in months") ~ C(ethnicity) + C(Q("controlling case sentencing county")) + count_current_case_enhancement + count_current_crimes_against_persons + count_current_drug_crimes + count_current_other_crimes + count_current_property_crimes + count_current_total + count_current_enhancements + count_current_in_prison + count_prior_case_enhancement + count_prior_crimes_against_persons + count_prior_drug_crimes + count_prior_other_crimes + count_prior_property_crimes + count_prior_total + count_prior_enhancements + count_prior_in_prison


## Run Regression Model

Fits OLS regression using HC3 robust standard errors to account for heteroscedasticity.

In [7]:
model = smf.ols(formula, data=data).fit(cov_type='HC3')
print(model.summary())

                                    OLS Regression Results                                   
Dep. Variable:     Q("aggregate sentence in months")   R-squared:                       0.256
Model:                                           OLS   Adj. R-squared:                  0.255
Method:                                Least Squares   F-statistic:                     1.790
Date:                               Wed, 25 Feb 2026   Prob (F-statistic):              0.167
Time:                                       00:19:51   Log-Likelihood:            -3.2240e+05
No. Observations:                              45459   AIC:                         6.449e+05
Df Residuals:                                  45388   BIC:                         6.456e+05
Df Model:                                         70                                         
Covariance Type:                                 HC3                                         
                                                            

c:\Users\vgwin\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 73, but rank is 2
  warnings.warn('covariance of constraints does not have full '


## Extract Key Results

Displays the ethnicity coefficient (the disparity estimate) with confidence interval.

In [8]:
ethnicity_coef = model.params[f'C(ethnicity)[T.{unexposed}]']
ethnicity_pval = model.pvalues[f'C(ethnicity)[T.{unexposed}]']
ethnicity_ci = model.conf_int().loc[f'C(ethnicity)[T.{unexposed}]']

print(f"\n{'='*60}")
print(f"DISPARITY ESTIMATE")
print(f"{'='*60}")
print(f"Comparison: {unexposed} vs {exposed} (reference)")
print(f"\nAdjusted Difference: {ethnicity_coef:.2f} months")
print(f"95% Confidence Interval: [{ethnicity_ci[0]:.2f}, {ethnicity_ci[1]:.2f}]")
print(f"P-value: {ethnicity_pval:.4f}")
print(f"\nInterpretation: After controlling for offense type, criminal history,")
print(f"enhancements, and county, {unexposed} defendants receive on average")
print(f"{ethnicity_coef:.2f} months {'longer' if ethnicity_coef > 0 else 'shorter'} sentences than {exposed} defendants.")
print(f"{'='*60}")


DISPARITY ESTIMATE
Comparison: Black vs White (reference)

Adjusted Difference: 4.54 months
95% Confidence Interval: [1.34, 7.73]
P-value: 0.0053

Interpretation: After controlling for offense type, criminal history,
enhancements, and county, Black defendants receive on average
4.54 months longer sentences than White defendants.


## Model Diagnostics

Checks for autocorrelation, heteroscedasticity, and multicollinearity.

In [9]:
dw = durbin_watson(model.resid)
print(f"Durbin-Watson: {dw:.3f}")
print(f"  Interpretation: {'No significant autocorrelation' if 1.5 < dw < 2.5 else 'Potential autocorrelation detected'}")

bp_test = het_breuschpagan(model.resid, model.model.exog)
bp_stat, bp_pval = bp_test[0], bp_test[1]
print(f"\nBreusch-Pagan Test: p-value = {bp_pval:.4f}")
print(f"  Interpretation: {'No significant heteroscedasticity' if bp_pval > 0.05 else 'Heteroscedasticity detected (HC3 standard errors already applied)'}")

print(f"\nR-squared: {model.rsquared:.3f}")
print(f"Adjusted R-squared: {model.rsquared_adj:.3f}")
print(f"\nSample size: {len(data):,}")
print(f"Number of predictors: {len(model.params) - 1}")

Durbin-Watson: 1.738
  Interpretation: No significant autocorrelation

Breusch-Pagan Test: p-value = 0.0000
  Interpretation: Heteroscedasticity detected (HC3 standard errors already applied)

R-squared: 0.256
Adjusted R-squared: 0.255

Sample size: 45,459
Number of predictors: 73
